# Pulsed-SABRE: Analytical model for a three-spin system
In this notebook, we will derive an analytical model for a three-spin system that is subjected to a pulsed polarisation transfer field. We consider a cosine-modulated field and derive a time-independent representation of the Hamiltonian using rotating frame transformations, Floquet theory, and average Hamiltonian theory. Finally, we come up with a LAC condition that has been generalised for the pulsed polarisation transfer fields and show how the effective coupling strengths can be modified. The appearance of several LACs is demonstrated by plotting the energies obtained from numerical diagonalisation of the Floquet matrix.

The calculations are made using SymPy and NumPy and the plotting is performed using Matplotlib.

In [ ]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt

### Split the Hamiltonian into components of Pauli matrices
We start from the previously calculated three-spin Hamiltonian that was written in the singlet-triplet Zeeman basis. We further truncate the state space by only considering the states of interest (either $\ket{S\alpha}$ and $\ket{T_+ \beta}$, or $\ket{S\beta}$ and $\ket{T_- \alpha}$). This reduces the Hamiltonian into $2 \times 2$ blocks, which are decomposed into components of Pauli matrices as follows:
$$
\hat{H} = s_0\hat{\sigma}_0 + s_x\hat{\sigma}_x + s_y\hat{\sigma}_y + s_z\hat{\sigma}_z
$$
In the following block of code, we will perform a sanity check: We use the decomposition to calculate back the original $2 \times 2$ Hamiltonians to verify that the decomposition is correct.

In [ ]:
s0, sx, sy, sz = symbols("s_0, s_x, s_y, sz", real=True)
J12, J13, J23 = symbols("J_12, J_13, J_23", real=True)
w1, w2, w3 = symbols(r"\omega_1, \omega_2, \omega_3", real=True)

sigma_0 = Matrix([
    [1, 0],
    [0, 1]
])

sigma_x = Matrix([
    [0, 1],
    [1, 0]
])

sigma_y = Matrix([
    [0, -I],
    [I, 0]
])

sigma_z = Matrix([
    [1, 0],
    [0, -1]
])

H = s0*sigma_0 + sx*sigma_x +sy*sigma_y + sz*sigma_z
display(H)

H1 = H.subs([
    (s0, (- pi*(2*J12 + J13 + J23) + (w1 + w2) ) / 4),
    (sx, pi * (J23 - J13) / sqrt(2)),
    (sy, 0),
    (sz, (- pi*(4*J12 - J13 - J23) + (-w1 - w2 + 2*w3) ) / 4)
])
display(simplify(H1))

H2 = H.subs([
    (s0, (- pi*(2*J12 + J13 + J23) - (w1 + w2) ) / 4),
    (sx, - pi * (J23 - J13) / sqrt(2)),
    (sy, 0),
    (sz, (- pi*(4*J12 - J13 - J23) - (-w1 - w2 + 2*w3) ) / 4)
])
display(simplify(H2))

### Construct the Hamiltonian for cosine-modulated pulses
In this discussion, we use a cosine-modulated magnetic field $B(t) = B_0 + B_1\cos(\omega_p t)$. This means that the coefficient $s_z$ becomes time-dependent. We split the coefficient $s_z$ into a time-independent part $s_{0z}$ and time-dependent part $s_{1z}(t) = s_{1z}\cos(\omega_p t)$. In addition, we discard the identity component $s_0\hat{\sigma}_0$, as it results only in a global phase shift. The Hamiltonian becomes:
$$
\hat{H} = s_x\hat{\sigma}_x + (s_{0z} + s_{1z}\cos(\omega_p t))\hat{\sigma}_z
$$

In [ ]:
s0z, s1z = symbols("s_0z, s_1z", real=True)
wp, t = symbols(r"\omega_p, t", positive=True)

H = sx*sigma_x + (s0z + s1z*cos(wp*t))*sigma_z
display(H)

### Transform the Hamiltonian into a rotating frame
We compute an operator for a unitary transformation that transforms the Hamiltonian into a rotating frame defined by the time-dependent term $s_{1z}\cos(\omega_p t)\hat{\sigma}_z$. The unitary operator is:
$$
\hat{U}_1 = e^{-is_{1z}\hat{\sigma}_z \int_0^t \cos(\omega_p t')dt'}
$$
The unitary transformation is then applied to the Hamiltonian by:
$$
\tilde{H} = \hat{U}_1^{\dag}\hat{H}\hat{U}_1 - i\hat{U}_1^{\dag}\dot{\hat{U}}_1
$$

In [ ]:
tp = symbols(r"t'", positive=True)

U1 = exp(-I*s1z*sigma_z*integrate(cos(wp*tp), (tp, 0, t)))
display(U1)

Hr1 = U1.H * H * U1 - I*U1.H*diff(U1, t)
Hr1 = simplify(Hr1)
display(Hr1)

### Perform the Jacobi-Anger expansion
We rewrite the off-diagonal terms using the Jacobi-Anger expansion:
$$
e^{iz\sin\theta} = \sum_{n=-\infty}^{\infty} J_n(z)e^{in\theta}
$$
In our case, $z = 2s_{1z}/\omega_p$ and $\theta = \omega_p t$.

In [ ]:
n = symbols("n", integer=True)

jacobi_anger = Sum(besselj(n, 2*s1z/wp) * exp(I*n*wp*t), (n, -oo, oo))
jacobi_anger_conj = Sum(besselj(n, 2*s1z/wp) * exp(-I*n*wp*t), (n, -oo, oo))
Hr1 = Matrix(Hr1)
Hr1[0, 1] = sx*jacobi_anger
Hr1[1, 0] = sx*jacobi_anger_conj
display(Hr1)

### Analyse the Hamiltonian using Floquet theory
In the previous work on pulsed polarisation transfer fields, the effective time-independent Hamiltonian was obtained by using the average Hamiltonian theory at this stage to the first order. From the off-diagonal elements, only the term $n=0$ survives. However, this Hamiltonian does not depict the full picture, as it only applies to the case where $s_{0z} \approx 0$. In fact, the higher-order terms involving the commutators in the average Hamiltonian theory become significant if $s_{0z} \gg 0$.

Instead, we may interpret the Hamiltonian using the Floquet theory, where the time-dependent Hamiltonian is expressed as a time-independent Hamiltonian in a (infinite-dimensional) product space $\mathcal{H} \otimes \mathcal{F}$:
$$
\Gamma =
\begin{bmatrix}
\ddots & \vdots                     &                  &                             &         \\
\cdots & \tilde{H}^{(0)} - \omega_p & \tilde{H}^{(-1)} & \tilde{H}^{(-2)}            &         \\
       & \tilde{H}^{(1)}            & \tilde{H}^{(0)}  & \tilde{H}^{(-1)}            &         \\
       & \tilde{H}^{(2)}            & \tilde{H}^{(1)}  & \tilde{H}^{(0)} + \omega_p  & \cdots  \\
       &                            &                  & \vdots                      & \ddots
\end{bmatrix}
$$
Here, the $\tilde{H}^{(n)}$ are the terms from the Fourier expansion of our Hamiltonian:
$$
\tilde{H} = \sum_n \tilde{H}^{(n)}e^{in\omega_p t}
$$

For demonstration purposes, we will construct the $5 \times 5$ Floquet matrix.

In [ ]:
H0 = Matrix([
    [s0z,                   sx*besselj(0, 2*s1z/wp)],
    [sx*besselj(0, 2*s1z/wp),  -s0z]
])
Hn = Matrix([
    [0,                     sx*besselj(n, 2*s1z/wp)],
    [sx*besselj(-n, 2*s1z/wp),  0]
])
Z = Matrix([
    [0, 0],
    [0, 0]
])
Id = Matrix([
    [1, 0],
    [0, 1]
])

floquet = BlockMatrix([
    [H0 - 2*wp*Id,      Hn.subs(n,-1),      Hn.subs(n,-2),  Hn.subs(n,-3),      Hn.subs(n,-4)],
    [Hn.subs(n,1),      H0 - wp * Id,       Hn.subs(n,-1),  Hn.subs(n,-2),      Hn.subs(n,-3)],
    [Hn.subs(n,2),      Hn.subs(n,1),       H0,             Hn.subs(n,-1),      Hn.subs(n,-2)],
    [Hn.subs(n,3),      Hn.subs(n,2),       Hn.subs(n,1),   H0 + wp*Id,         Hn.subs(n,-1)],
    [Hn.subs(n,4),      Hn.subs(n,3),       Hn.subs(n,2),   Hn.subs(n,1),       H0 + 2*wp*Id]
])
floquet = Matrix(floquet)

display(floquet)

### Generalised LAC condition and transformation to another rotating frame
We remind ourselves that a coherent transfer between two states can occur efficiently when their diagonal elements become equal. Now, the Floquet matrix reveals a generalised LAC condition, as the diagonal elements may become equal when:
$$
2s_{0z} + m\omega_p = 0
$$
In other words, when that condition is fulfilled, two Fourier modes will be strongly coupled which results in coherent polarisation transfer with a frequency determined by the off-diagonal element connecting the states. In this case, the two states will be coupled by a coupling strength $J_m\left(2s_{1z} / \omega_p\right)$.

To come up with a time-independent $2 \times 2$ Hamiltonian, we will apply another rotating frame transformation inspired by the Floquet theory. We define $m$ to correspond to the Floquet mode for which $2s_{0z} + m\omega_p \approx 0$. The unitary operator of interest is:
$$
\hat{U} = e^{\frac{i\hat{\sigma}_z}{2}m \omega_p t}
$$
The transformation is applied again by:
$$
\tilde{\tilde{H}} = \hat{U}_2^{\dag}\tilde{H}\hat{U}_2 - i\hat{U}_2^{\dag}\dot{\hat{U}}_2
$$
The result is manually simplified, as SymPy does not multiply the exponential term inside the sum.

In [ ]:
m = symbols("m", integer=True)

U2 = exp(Rational(1/2)*I*sigma_z*m*wp*t)
display(U2)

Hr2 = U2.H * Hr1 * U2 - I*U2.H*diff(U2, t)
Hr2 = simplify(Hr2)
display(Hr2)

# Manually simplify the result
outside_01 = Hr2[0,1].args[0] * Hr2[0,1].args[2]
sum_part_01 = Hr2[0,1].args[1]

outside_10 = Hr2[1,0].args[0] * Hr2[1,0].args[2]
sum_part_10 = Hr2[1,0].args[1]

Hr2 = Matrix(Hr2)
Hr2[0,1] = Sum(simplify(outside_01 * sum_part_01.function), *sum_part_01.limits)
Hr2[1,0] = Sum(simplify(outside_10 * sum_part_10.function), *sum_part_10.limits)

display(Hr2)

### Apply the average Hamiltonian theory
We calculate the first-order term of the average Hamiltonian theory:
$$
\tilde{\tilde{H}}^{(1)}_{\mathrm{eff}} = \frac{1}{T} \int_{0}^{T} \tilde{\tilde{H}} dt
$$
Here, period of the pulse is $T = 2\pi / \omega_p$. From the off-diagonal terms, we find that only the term $n = m$ survives. SymPy is incapable of performing the integral for the off-diagonal elements, so we will write them manually.

In [ ]:
Hr2_eff = 1 / (2*pi/wp) * integrate(Hr2, (t, 0, 2*pi/wp))
display(Hr2_eff)

Hr2_eff = Matrix(Hr2_eff)
Hr2_eff[0,1] = besselj(m, 2*s1z/wp)
Hr2_eff[1,0] = besselj(m, 2*s1z/wp)
display(Hr2_eff)

### Calculating the LAC condition
We arrive to a time-independent effective Hamiltonian that was predicted by Floquet theory. There will be a LAC when the diagonal elements are equal. We calculate the $B_0$ for the LAC condition. The condition is calculated for transitions $\ket{S\alpha} \rightarrow \ket{T_+\beta}$ and $\ket{S\beta} \rightarrow \ket{T_-\alpha}$.

In [ ]:
B0 = symbols("B_0", real=True)
y1, y2, y3 = symbols(r"\gamma_1, \gamma_2, \gamma_3", real=True)
d1, d2, d3 = symbols(r"\delta_1, \delta_2, \delta_3", real=True)
J12, J13, J23 = symbols("J_12, J_13, J_23", real=True)

eq_LAC = Hr2[0,0] - Hr2[1,1]

eq_LAC1 = eq_LAC.subs(s0z,(- pi*(4*J12 - J13 - J23) + (-w1 - w2 + 2*w3) ) / 4) \
                .subs(w1, -y1*B0*(1+d1)) \
                .subs(w2, -y2*B0*(1+d2)) \
                .subs(w3, -y3*B0*(1+d3))
B0_LAC1 = solve(eq_LAC1, B0)[0]
display(B0_LAC1)

eq_LAC2 = eq_LAC.subs(s0z,(- pi*(4*J12 - J13 - J23) - (-w1 - w2 + 2*w3) ) / 4) \
                .subs(w1, -y1*B0*(1+d1)) \
                .subs(w2, -y2*B0*(1+d2)) \
                .subs(w3, -y3*B0*(1+d3))
B0_LAC2 = solve(eq_LAC2, B0)[0]
display(B0_LAC2)

### Example LACs: 3-fluoropyridine
We calculate example values for a 3-fluoropyridine system, where the $J$-couplings are $J_{12} = -7.67\ \mathrm{Hz}$, $J_{13} = -0.05\ \mathrm{Hz}$, and $J_{23} = 2.08\ \mathrm{Hz}$, and the chemical shifts are $\delta_1 = \delta_2 = -23.1\ \mathrm{ppm}$ and $\delta_3 = -124.2\ \mathrm{ppm}$. The period of the cosine wave is chosen to be $T = 10\ \mathrm{ms}$. The LACs are calculated for cases $m=0$ and $m=1$. It is observed that the case $m=0$ reduces to the static field LAC condition.

In [ ]:
B0_LAC2_m0 = B0_LAC2.subs([
    (J12, -7.67),
    (J13, -0.05),
    (J23, 2.08),
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
    (pi, np.pi),
    (m, 0)
])
display(B0_LAC2_m0)

B0_LAC2_m1 = B0_LAC2.subs([
    (J12, -7.67),
    (J13, -0.05),
    (J23, 2.08),
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
    (pi, np.pi),
    (m, 1)
])
display(B0_LAC2_m1)

### Tuning the coupling strengths by adjusting $B_1$
Once the desired LAC condition has been established, we may tune the coupling strengths by adjusting the pulse amplitude $B_1$. For example, in the case of $m=1$, the bessel function $J_1(2s_{1z}/\omega_p)$ reaches its maximum value of $\sim 0.58$ when the argument is $2s_{1z}/\omega_p = 1.84$. We solve for $B_1$ that results in the desired value.

In [ ]:
B1 = symbols("B_1", real=True)

eq = 2*s1z/wp - 1.84
eq = eq.subs(s1z, (-B1 * (y1*(1+d1) + y2*(1+d2) -2*y3*(1+d3)))/4)

B1_example = solve(eq, B1)[0]
B1_example = B1_example.subs([
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
])
display(B1_example)

### Numerical diagonalisation of the Floquet matrix
Next, we demonstrate the appearance of the higher-order energy level anticrossings by performing the numerical diagonalisation of the above Floquet matrix, truncated to include terms up to $m = \pm 5$, with the example 3-fluoropyridine system. The numerical diagonalisation is performed with static offset field values of $B_0 = 0 \ldots 500\ \mathrm{\mu T}$. The pulse amplitude is set arbitrarily to $B_1 = 100\ \mathrm{\mu T}$.

The resulting plot demonstrates the appearance of the LACs between different Fourier modes from $m=0$ up to $m=5$. If we increased $m$, more LACs would appear.

In [ ]:
m_max = 5

floquet = [[None for _ in range(2*m_max+1)] for _ in range(2*m_max+1)]

for n1 in range(-m_max, m_max+1):
    for n2 in range(-m_max, m_max+1):

        if n1 == n2:
            floquet[n1+N][n2+N] = H0 + n1*wp*Id

        else:
            floquet[n1+N][n2+N] = Hn.subs(n, n1-n2)

floquet = Matrix(BlockMatrix(floquet))

floquet = floquet.subs([
    (sx, pi * (J23 - J13) / sqrt(2)),
    (s0z, (- pi*(4*J12 - J13 - J23) - B0 * (y1*(1+d1) + y2*(1+d2) - 2*y3*(1+d3)) ) / 4),
    (s1z, (-B1 * (y1*(1+d1) + y2*(1+d2) -2*y3*(1+d3)))/4)
])

floquet = floquet.subs([
    (J12, -7.67),
    (J13, -0.05),
    (J23, 2.08),
    (y1, 267522188),
    (y2, 267522188),
    (y3, 251762483),
    (d1, -23.1*1e-6),
    (d2, -23.1*1e-6),
    (d3, -124.2*1e-6),
    (wp, 2*np.pi / (0.010)),
    (B1, 100e-6),
    (pi, np.pi)
]).evalf()

B0s = np.array([i for i in range(500)]) * 1e-6

Fs = [np.array(floquet.subs(B0, B0s[i])).astype(np.float64) for i in range(len(B0s))]
Fs = np.array(Fs)

eigvals = np.array([np.linalg.eigvals(F) for F in Fs]).real
eigvals = np.array([np.sort(eigval) for eigval in eigvals])

plt.plot(B0s*1e6, eigvals[:,2*m_max])
plt.plot(B0s*1e6, eigvals[:,2*m_max+1])
plt.xlabel(r"$B_0$ (µT)")
plt.ylabel("Energy (rad/s)")
plt.show()

np.savetxt("eigvals.txt", eigvals[:, 2*m_max:2*m_max+2])